# Setup Text Viewer - Rationale & Reasoning Deep-Dive

**Purpose:** Read the full text, rationale, and reasoning for AI-generated experimental setups

---

## What This Notebook Provides:

- 📝 **Rationale for each component** - Why AI chose each baseline, metric, dataset
- 📊 **Side-by-side three-way comparison** - Non-Agentic vs Agentic vs Ground Truth
- 🔍 **Overlap analysis** - What's shared vs unique between approaches
- 🔧 **Full JSON view** - Complete data structure for debugging
- 📖 **Redacted paper context** - What the AI actually saw

---

## When to Use This Notebook:

**Use this when you want to:**
- Read the reasoning behind AI's choices
- See rationale for why each component was selected
- Compare all three approaches side-by-side in a table
- Understand diversity between approaches

**For quantitative analysis** (scores, charts, alignment, costs), use:
- **`visualize_results.ipynb`** - Single-paper analysis with scores/charts
- **`Analysis.ipynb`** - Multi-paper statistical analysis

---

## 1. Setup and Configuration

In [1]:
# ============================================================================
# CONFIGURATION
# ============================================================================

# Path to your run directory
RUN_FOLDER = "../runs/20251214_214006_full_eval"

# Option 1: Set PAPER_ID directly (fastest)
PAPER_ID = None  # or set to None to list papers

# Option 2: Set PAPER_INDEX if PAPER_ID is None (will list papers first)
PAPER_INDEX = 0  # Set to 0 for first paper, 1 for second, etc. (only used if PAPER_ID is None)

# ============================================================================


In [2]:
# Import required libraries
import sys
import json
import textwrap

# Import our utilities
from analysis_utils import (
    # Data loading
    list_papers_in_run,
    load_single_paper_data,
    
    # Data extraction
    get_paper_context,
    extract_setup_components,
    
    # Text formatting (with rationale)
    format_component_with_rationale,
    create_side_by_side_comparison,
    compute_overlap_analysis,
    
    # Display
    print_section_header,
    
    # Constants
    APPROACH_NON_AGENTIC,
    APPROACH_AGENTIC
)

print("✓ All libraries and utilities imported successfully")

✓ All libraries and utilities imported successfully


## 2. Select Paper to Analyze

In [3]:
# Select paper - either from config or by index
if PAPER_ID is None:
    # Need to select - list all papers
    papers = list_papers_in_run(RUN_FOLDER)
    
    if not papers:
        print("❌ No papers found in run. Check RUN_FOLDER path.")
        sys.exit(1)
    
    print("Available papers:")
    for i, paper in enumerate(papers):
        print(f"{i}: {paper['id'][:80]}")
    
    # Use PAPER_INDEX from configuration
    if len(papers) == 1:
        PAPER_INDEX = 0
        print(f"\n✓ Automatically selected the only paper (index 0)")
    else:
        # Validate PAPER_INDEX
        if PAPER_INDEX < 0 or PAPER_INDEX >= len(papers):
            print(f"\n❌ Invalid PAPER_INDEX={PAPER_INDEX}. Must be 0-{len(papers)-1}")
            print(f"   Please update PAPER_INDEX in the configuration cell above.")
            sys.exit(1)
        print(f"\n✓ Using PAPER_INDEX={PAPER_INDEX} from configuration")
    
    PAPER_ID = papers[PAPER_INDEX]['id']
else:
    # PAPER_ID already set in configuration
    print(f"Using PAPER_ID from configuration")

print(f"\nSelected: {PAPER_ID}\n")


Available papers:
0: Playing Atari with Deep Reinforcement Learning_2319a491
1: Deep Reinforcement Learning from Human Preferences_5bbb6f9a
2: Asynchronous Methods for Deep Reinforcement Learni_69e76e16

✓ Using PAPER_INDEX=0 from configuration

Selected: Playing Atari with Deep Reinforcement Learning_2319a491



## 3. Load Paper Data

In [4]:
# Load the selected paper's data
results, context = load_single_paper_data(RUN_FOLDER, PAPER_ID)

if not results:
    print("❌ Failed to load paper data")
    sys.exit(1)

# Extract paper context
paper_context = get_paper_context(context)

# Extract setup components
non_agentic_setup = results.get(APPROACH_NON_AGENTIC, {})
agentic_setup = results.get(APPROACH_AGENTIC, {})

non_agentic_components = extract_setup_components(non_agentic_setup)
agentic_components = extract_setup_components(agentic_setup)

# Extract ground truth if available
ground_truth = results.get('ground_truth', {})
ground_truth_components = None

if ground_truth and ground_truth.get('has_experiments'):
    ground_truth_components = {
        'baselines': ground_truth.get('baselines', []),
        'metrics': ground_truth.get('metrics', []),
        'datasets': ground_truth.get('datasets', [])
    }

print("✓ Data loaded successfully")
print(f"  Ground truth available: {'✓' if ground_truth_components else '✗'}\n")

✓ Data loaded successfully
  Ground truth available: ✓



## 4. Redacted Paper Context

**This is what the AI saw** (experimental sections were removed):

In [5]:
print_section_header("REDACTED PAPER CONTEXT (What AI Saw)", width=80)

print("TITLE:")
print(textwrap.fill(paper_context['title'], width=80))
print()

print("=" * 80)
print("DOMAIN:")
print("=" * 80)
print(paper_context['domain'])
print()

print("=" * 80)
print("RESEARCH QUESTION:")
print("=" * 80)
print(textwrap.fill(paper_context['research_question'], width=80))
print()

if context and context.get('abstract'):
    print("=" * 80)
    print("ABSTRACT (excerpt):")
    print("=" * 80)
    abstract = context['abstract'][:500] + "..." if len(context['abstract']) > 500 else context['abstract']
    print(textwrap.fill(abstract, width=80))
    print()

print("=" * 80)
print("NOTE: Experimental sections (experiments, results, evaluation) were REDACTED.")
print("The AI never saw the actual baselines, metrics, or datasets used.")
print("=" * 80)
print()


REDACTED PAPER CONTEXT (What AI Saw)

TITLE:
playing atari with deep reinforcement learning

DOMAIN:
RL

RESEARCH QUESTION:
The main research question of this paper is: Can a deep learning model,
specifically a convolutional neural network trained with a variant of
Q-learning, successfully learn control policies directly from high-dimensional
sensory input (raw pixels) and outperform previous approaches on Atari 2600
games?

ABSTRACT (excerpt):
 We present the first deep learning model to successfully learn control policies
directly from high-dimensional sensory input using reinforcement learning. The
model is a convolutional neural network, trained with a variant of Q-learning,
whose input is raw pixels and whose output is a value function estimating future
rewards. We apply our method to seven Atari 2600 games from the Arcade Learning
Environment, with no adjustment of the architecture or learning algorithm. We
find that it outperfor...

NOTE: Experimental sections (experiments, res

## 5. Non-Agentic Generated Setup

**Single-shot generation** with rationale for each component:

In [6]:
print_section_header("NON-AGENTIC GENERATED SETUP", width=80)

print("BASELINES:")
print("─" * 80)
print(format_component_with_rationale(non_agentic_components['baselines'], 'baseline'))

print("=" * 80)
print("METRICS:")
print("─" * 80)
print(format_component_with_rationale(non_agentic_components['metrics'], 'metric'))

print("=" * 80)
print("DATASETS:")
print("─" * 80)
print(format_component_with_rationale(non_agentic_components['datasets'], 'dataset'))

# Show metadata
non_agentic_meta = non_agentic_setup.get('setup', {}).get('metadata', {})
if not non_agentic_meta:
    non_agentic_meta = results.get(APPROACH_NON_AGENTIC, {}).get('metadata', {})

print("=" * 80)
print("METADATA:")
print("─" * 80)
print(f"Time elapsed: {non_agentic_meta.get('time_elapsed', 0):.2f}s")
print(f"LLM calls: {non_agentic_meta.get('llm_calls', 0)}")
print("=" * 80)
print()


NON-AGENTIC GENERATED SETUP

BASELINES:
────────────────────────────────────────────────────────────────────────────────
1. TD-Gammon
   Description: A backgammon-playing program that learned entirely by reinforcement learning and self-play
   Rationale: This is a classic RL success story mentioned in the Related Work section, so it should be included as a baseline.

2. Q-learning with non-linear function approximation
   Description: A model-free RL algorithm that uses a non-linear function approximator like a neural network to represent the Q-function
   Rationale: The Related Work section mentions that combining Q-learning with non-linear function approximators can be challenging, so this is an important baseline to include.

3. Random Agent
   Description: An agent that selects actions randomly
   Rationale: A simple random baseline is important to establish a lower bound on performance.

METRICS:
────────────────────────────────────────────────────────────────────────────────
1. 

## 6. Agentic Generated Setup

**Multi-draft generation with selection** and rationale:

In [7]:
print_section_header("AGENTIC GENERATED SETUP", width=80)

print("BASELINES:")
print("─" * 80)
print(format_component_with_rationale(agentic_components['baselines'], 'baseline'))

print("=" * 80)
print("METRICS:")
print("─" * 80)
print(format_component_with_rationale(agentic_components['metrics'], 'metric'))

print("=" * 80)
print("DATASETS:")
print("─" * 80)
print(format_component_with_rationale(agentic_components['datasets'], 'dataset'))

# Show metadata
agentic_meta = agentic_setup.get('setup', {}).get('metadata', {})
if not agentic_meta:
    agentic_meta = results.get(APPROACH_AGENTIC, {}).get('metadata', {})

print("=" * 80)
print("METADATA:")
print("─" * 80)
print(f"Time elapsed: {agentic_meta.get('time_elapsed', 0):.2f}s")
print(f"LLM calls: {agentic_meta.get('llm_calls', 0)}")

# Show exploration tree if available
exploration_tree = agentic_setup.get('setup', {}).get('exploration_tree', [])
if exploration_tree:
    print("\nEXPLORATION TREE:")
    for draft in exploration_tree:
        draft_id = draft.get('draft_id', 'Unknown')
        score = draft.get('score', 0)
        is_best = draft.get('is_best', False)
        marker = "⭐ SELECTED" if is_best else ""
        print(f"  Draft {draft_id}: Score = {score:.3f} {marker}")

print("=" * 80)
print()


AGENTIC GENERATED SETUP

BASELINES:
────────────────────────────────────────────────────────────────────────────────
1. DQN (Mnih et al. 2015)
   Description: Deep Q-Network using convolutional neural networks to learn directly from pixel inputs
   Rationale: This represents the state-of-the-art deep reinforcement learning approach for Atari games, which the proposed model aims to improve upon.

2. Prioritized Dueling DQN (Wang et al. 2016)
   Description: An extension of DQN with prioritized experience replay and a dueling network architecture
   Rationale: This is a more advanced deep RL model that has shown improved performance on Atari games compared to the original DQN.

3. Human expert
   Description: Performance of skilled human players on the Atari 2600 games
   Rationale: Comparing to human-level performance is an important benchmark for evaluating the deep learning model's capabilities.

METRICS:
───────────────────────────────────────────────────────────────────────────────

## 7. Ground Truth (What Authors Actually Did)

**This was REDACTED** - extracted from experimental sections that were removed:

In [8]:
if ground_truth_components:
    print_section_header("GROUND TRUTH - WHAT WAS REDACTED (AI DID NOT SEE THIS)", width=80)
    
    print("These experimental details were removed before AI saw the paper.")
    print("This is what the authors actually used in their experiments.\n")
    
    print("=" * 80)
    print("ACTUAL BASELINES USED:")
    print("─" * 80)
    baselines = ground_truth_components['baselines']
    if baselines:
        print(format_component_with_rationale(baselines, 'baseline'))
    else:
        print("No baselines extracted (paper may not have explicit baseline comparisons)\n")
    
    print("=" * 80)
    print("ACTUAL METRICS USED:")
    print("─" * 80)
    metrics = ground_truth_components['metrics']
    if metrics:
        print(format_component_with_rationale(metrics, 'metric'))
    else:
        print("No metrics extracted\n")
    
    print("=" * 80)
    print("ACTUAL DATASETS USED:")
    print("─" * 80)
    datasets = ground_truth_components['datasets']
    if datasets:
        print(format_component_with_rationale(datasets, 'dataset'))
    else:
        print("No datasets extracted\n")
    
    print("=" * 80)
    print("EXTRACTION METADATA:")
    print("─" * 80)
    print(f"Confidence: {ground_truth.get('extraction_confidence', 'unknown').upper()}")
    if ground_truth.get('notes'):
        print(f"\nNotes:")
        print(textwrap.fill(ground_truth['notes'], width=80))
    print()
    
    print("=" * 80)
    print("NOTE: This ground truth was extracted from experimental sections")
    print("      that were REDACTED before the AI saw the paper.")
    print("=" * 80)
    print()
else:
    print_section_header("NO GROUND TRUTH AVAILABLE", width=80)
    print("Either no experiments were found in the paper, or ground truth")
    print("extraction failed. The AI still generated setups based on the")
    print("research context alone.")
    print("=" * 80)
    print()


GROUND TRUTH - WHAT WAS REDACTED (AI DID NOT SEE THIS)

These experimental details were removed before AI saw the paper.
This is what the authors actually used in their experiments.

ACTUAL BASELINES USED:
────────────────────────────────────────────────────────────────────────────────
1. Previous reinforcement learning algorithms
   Description: The paper states that their approach outperforms all previous reinforcement learning algorithms on six of the seven games attempted.

ACTUAL METRICS USED:
────────────────────────────────────────────────────────────────────────────────
1. Performance
   Description: The paper measures how the model performs compared to a human expert player.

ACTUAL DATASETS USED:
────────────────────────────────────────────────────────────────────────────────
1. Atari 2600 games
   Description: The paper evaluates the model on seven Atari 2600 games from the Arcade Learning Environment.

EXTRACTION METADATA:
──────────────────────────────────────────────────

## 8. Side-by-Side Comparison

**Three-way comparison** of Non-Agentic vs Agentic vs Ground Truth:

In [9]:
comparison_table = create_side_by_side_comparison(
    non_agentic_components,
    agentic_components,
    ground_truth_components
)

print(comparison_table)
print()

THREE-WAY SIDE-BY-SIDE COMPARISON

BASELINES:
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Non-Agentic                         | Agentic                             | Ground Truth (Actual)              
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
TD-Gammon                           | DQN (Mnih et al. 2015)              | Previous reinforcement learning al 
Q-learning with non-linear functio  | Prioritized Dueling DQN (Wang et a  |                                    
Random Agent                        | Human expert                        |                                    

METRICS:
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Non-Agentic                         | Agentic                             | Ground Truth (Actual)              
─────────────────────

## 9. Overlap Analysis

**What's shared vs unique** between the two AI approaches:

In [10]:
overlap_analysis = compute_overlap_analysis(
    non_agentic_components,
    agentic_components
)

print(overlap_analysis)
print()

OVERLAP ANALYSIS

BASELINES:
  Shared: 0
  Only Non-Agentic: 3
    - q-learning with non-linear function approximation
    - random agent
    - td-gammon
  Only Agentic: 3
    - dqn (mnih et al. 2015)
    - human expert
    - prioritized dueling dqn (wang et al. 2016)

METRICS:
  Shared: 1
    - sample efficiency
  Only Non-Agentic: 2
    - game score
    - human expert score
  Only Agentic: 3
    - convergence speed
    - mean human-normalized score
    - number of games where the agent exceeds human performance

DATASETS:
  Shared: 0
  Only Non-Agentic: 1
    - atari 2600 games
  Only Agentic: 2
    - arcade learning environment
    - human gameplay recordings



## 10. Full JSON View (Optional)

**Raw JSON** for detailed inspection and debugging:

In [11]:
# Optional: Uncomment to see full JSON

# print("=" * 80)
# print("NON-AGENTIC SETUP (Full JSON):")
# print("=" * 80)
# non_agentic_full = non_agentic_setup.get('setup', {})
# if 'ai_setup' in non_agentic_full:
#     non_agentic_full = non_agentic_full['ai_setup']
# print(json.dumps(non_agentic_full, indent=2))
# print()

# print("=" * 80)
# print("AGENTIC SETUP (Full JSON):")
# print("=" * 80)
# agentic_full = agentic_setup.get('setup', {})
# if 'ai_setup' in agentic_full:
#     agentic_full = agentic_full['ai_setup']
# print(json.dumps(agentic_full, indent=2))
# print()

print("ℹ️  Uncomment the code above to view full JSON structures")

ℹ️  Uncomment the code above to view full JSON structures


## Summary

This notebook provides a text-focused view of the generated experimental setups, highlighting:

✅ **Rationale** - Why each component was chosen  
✅ **Side-by-Side** - Easy comparison of all three approaches  
✅ **Overlap** - Understanding diversity between approaches  
✅ **Context** - What the AI saw vs what was redacted  

For **quantitative analysis** with scores, charts, and recommendations, use:
- **`visualize_results.ipynb`** for single-paper analysis
- **`Analysis.ipynb`** for multi-paper statistics